# Qwen3.6-35B-A3B • TurboQuant • ASI-Evolve  (Colab Pro / G4 Blackwell 98GB)

End-to-end: download Qwen3.6, convert + Q5_K_M quantize, build QuantClaw (+ patched upstream llama.cpp) with CUDA, start `llama-server`, start the QuantClaw gateway (which spawns the ASI-Evolve Python bridge as its sidecar), then fire an `evolve_start` tool call over the gateway's WebSocket RPC.

**Before running:** edit the `CONFIG` cell with:
1. `QUANTCLAW_REPO` — clone URL for your QuantClaw fork (private: use `https://<PAT>@github.com/...`)
2. `ASI_EVOLVE_REPO` — clone URL for the ASI-Evolve repo (holds `bridge_script`)
3. `BRIDGE_SCRIPT_REL` — path to the Python bridge entrypoint inside the ASI-Evolve repo
4. Colab secret `HF_TOKEN` — HuggingFace token with access to `Qwen/Qwen3.6-35B-A3B`

**Expected runtime on G4 Blackwell 98GB:** ~15 min download, ~10 min convert+quant, ~8 min build.

**llama.cpp distribution model:** upstream llama.cpp is *not* vendored or submodule'd into QuantClaw. Instead, `patches/turboquant/` in the QuantClaw repo holds a pinned upstream commit SHA and a unified patch. The clone cell clones upstream at that SHA into `ui/llama.cpp/llama.cpp/` and applies the patch.

**Facts the notebook encodes (verified against the source tree):**
- Built executable: `quantclaw` (single binary). Gateway launches via `quantclaw gateway run --config <json>`.
- Gateway IPC: **WebSocket** on `config.gateway.port`. Handshake = `connect.hello` (auth token + scopes), then JSON frames `{type:'req', id, method, params}`.
- Tools execute through the `chain.execute` RPC with a `{name, steps:[{tool, arguments}]}` body.
- Evolve is enabled by setting `config.evolve.enabled = true` and providing `bridge_script`; the gateway spawns the Python bridge via `EvolveRuntime::StartSidecar`.

## 0. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!nvcc --version | tail -2
!cat /etc/os-release | grep PRETTY_NAME
!free -h | head -2

## 1. CONFIG  edit this cell

In [ ]:
import os, pathlib, subprocess, secrets
from google.colab import userdata

# --- EDIT ---
QUANTCLAW_REPO    = 'https://github.com/J-mazz/QuantClaw.git'
QUANTCLAW_REF     = 'main'
ASI_EVOLVE_REPO   = 'https://github.com/J-mazz/ASI-Evolve.git'
ASI_EVOLVE_REF    = 'main'
BRIDGE_SCRIPT_REL = 'bridge/evolve_bridge.py'     # path inside ASI-Evolve repo
HF_MODEL_ID       = 'Qwen/Qwen3.6-35B-A3B'
QUANT_TYPE        = 'Q5_K_M'
CTX_SIZE          = 131072                        # 128K
SERVER_PORT       = 39201
GATEWAY_PORT      = 7777
AUTH_TOKEN        = secrets.token_hex(16)         # fresh token per session
# ------------

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

ROOT          = pathlib.Path('/content')
QC_DIR        = ROOT / 'QuantClaw'
EVOLVE_DIR    = ROOT / 'ASI-Evolve'
MODEL_DIR     = ROOT / 'models' / HF_MODEL_ID.split('/', 1)[1]
GGUF_F16      = MODEL_DIR / f'{MODEL_DIR.name}-f16.gguf'
GGUF_QUANT    = MODEL_DIR / f'{MODEL_DIR.name}-{QUANT_TYPE}.gguf'
SIDECAR_BIN   = MODEL_DIR / f'{MODEL_DIR.name}-{QUANT_TYPE}.turboquant.bin'
BUILD_DIR     = QC_DIR / 'build-colab'
BRIDGE_SCRIPT = EVOLVE_DIR / BRIDGE_SCRIPT_REL
GATEWAY_WS    = f'ws://127.0.0.1:{GATEWAY_PORT}/'

cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().split('\n')[0].replace('.', '')
CUDA_ARCH = cc or '80'
print(f'CUDA arch: sm_{CUDA_ARCH}')
print(f'Target: {HF_MODEL_ID}  {QUANT_TYPE}  ctx={CTX_SIZE}')
print(f'Auth token (session-scoped): {AUTH_TOKEN}')
for p in [QC_DIR, EVOLVE_DIR, MODEL_DIR, BUILD_DIR]:
    p.parent.mkdir(parents=True, exist_ok=True)

## 2. System deps

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y cmake ninja-build build-essential git-lfs pkg-config \
    libcurl4-openssl-dev nlohmann-json3-dev
git lfs install --skip-smudge
pip install -q 'huggingface_hub[cli]>=0.25' sentencepiece protobuf numpy requests tqdm websocket-client

## 3. Clone QuantClaw + ASI-Evolve

In [ ]:
# -- Guard against unedited CONFIG placeholders (silent bash redirection otherwise).
for name, val in [('QUANTCLAW_REPO', QUANTCLAW_REPO), ('ASI_EVOLVE_REPO', ASI_EVOLVE_REPO)]:
    if '<you>' in val or '<' in val or '>' in val:
        raise ValueError(f'{name} still contains placeholder: {val!r}  edit the CONFIG cell with a real git URL')

# 1) Clone QuantClaw (no submodules; llama.cpp is not vendored). Quote vars so any future placeholder errors surface from git, not bash.
!git clone --depth 1 --branch "{QUANTCLAW_REF}" "{QUANTCLAW_REPO}" "{QC_DIR}"
assert (QC_DIR / '.git').exists(), f'QuantClaw clone failed  check QUANTCLAW_REPO / QUANTCLAW_REF'

# 2) Clone ASI-Evolve (holds bridge_script).
!git clone --depth 1 --branch "{ASI_EVOLVE_REF}" "{ASI_EVOLVE_REPO}" "{EVOLVE_DIR}"
assert (EVOLVE_DIR / '.git').exists(), f'ASI-Evolve clone failed  check ASI_EVOLVE_REPO / ASI_EVOLVE_REF'

# 3) Resolve the pinned upstream SHA from the QuantClaw repo.
PATCH_DIR = QC_DIR / 'patches' / 'turboquant'
SHA_FILE  = PATCH_DIR / 'UPSTREAM_SHA'
if not SHA_FILE.exists():
    raise FileNotFoundError(
        f'{SHA_FILE} missing  commit + push `patches/turboquant/` (0001-turboquant.patch + UPSTREAM_SHA) '
        f'to {QUANTCLAW_REPO} before running this cell')
UPSTREAM_SHA = SHA_FILE.read_text().strip()
assert len(UPSTREAM_SHA) == 40 and all(c in '0123456789abcdef' for c in UPSTREAM_SHA), \
    f'UPSTREAM_SHA malformed: {UPSTREAM_SHA!r}'
patches = sorted(PATCH_DIR.glob('*.patch'))
assert patches, f'no *.patch files in {PATCH_DIR}'

# 4) Clone upstream llama.cpp at the pinned SHA.
LLAMA_DIR = QC_DIR / 'ui' / 'llama.cpp' / 'llama.cpp'
LLAMA_DIR.parent.mkdir(parents=True, exist_ok=True)
!git clone https://github.com/ggerganov/llama.cpp.git "{LLAMA_DIR}"
!git -C "{LLAMA_DIR}" checkout --quiet "{UPSTREAM_SHA}"

# 5) Apply the TurboQuant patch series (--check first so drift fails loudly).
import subprocess
for p in patches:
    print(f'applying {p.name}')
    subprocess.run(['git', '-C', str(LLAMA_DIR), 'apply', '--check', str(p)], check=True)
    subprocess.run(['git', '-C', str(LLAMA_DIR), 'apply',            str(p)], check=True)

# 6) Sanity asserts.
assert (LLAMA_DIR / 'CMakeLists.txt').exists(),         'upstream llama.cpp clone failed'
assert (LLAMA_DIR / 'src' / 'turboquant.cpp').exists(), 'turboquant.cpp missing after patch'
assert (LLAMA_DIR / 'src' / 'turboquant.h').exists(),   'turboquant.h missing after patch'
assert BRIDGE_SCRIPT.exists(), f'bridge_script not at {BRIDGE_SCRIPT}  fix BRIDGE_SCRIPT_REL'
print(f'OK  llama.cpp @ {UPSTREAM_SHA[:10]} + {len(patches)} patch(es)')

## 4. Download Qwen3.6-35B-A3B  (~70 GB BF16)

Skips `.bin` / `.pt` variants; keeps `.safetensors`.

In [ ]:
!huggingface-cli download {HF_MODEL_ID} \
    --local-dir {MODEL_DIR} \
    --local-dir-use-symlinks False \
    --exclude '*.bin' '*.pt' 'original/*'
!du -sh {MODEL_DIR}

## 5. HF  GGUF  (f16 intermediate)

The forked `convert_hf_to_gguf.py` maps Qwen3.5 hybrid arch to the `qwen35moe` code path. If Qwen3.6 ships a new `model_type` string, the converter will error with `unknown architecture`  patch the arch map in that file and re-run.

In [ ]:
CONVERT = QC_DIR / 'ui' / 'llama.cpp' / 'llama.cpp' / 'convert_hf_to_gguf.py'
REQ = CONVERT.parent / 'requirements' / 'requirements-convert_hf_to_gguf.txt'
!pip install -q -r {REQ}
!python {CONVERT} {MODEL_DIR} --outfile {GGUF_F16} --outtype f16
!ls -lh {GGUF_F16}

## 6. Build QuantClaw + llama.cpp (CUDA)

`quantclaw` is the single gateway/CLI binary. `llama-quantize` and `llama-server` come from the llama.cpp subproject.

In [ ]:
%%bash -s "$QC_DIR" "$BUILD_DIR" "$CUDA_ARCH"
set -e
QC_DIR=$1 ; BUILD_DIR=$2 ; ARCH=$3
cmake -S "$QC_DIR" -B "$BUILD_DIR" -G Ninja \
    -DCMAKE_BUILD_TYPE=Release \
    -DLLAMA_TURBOQUANT=ON \
    -DGGML_CUDA=ON \
    -DCMAKE_CUDA_ARCHITECTURES=$ARCH \
    -DLLAMA_BUILD_SERVER=ON
cmake --build "$BUILD_DIR" -j$(nproc) --target llama-quantize llama-server llama-cli quantclaw

## 7. Quantize  Q5_K_M

In [ ]:
QUANTIZE = BUILD_DIR / 'bin' / 'llama-quantize'
!{QUANTIZE} {GGUF_F16} {GGUF_QUANT} {QUANT_TYPE}
!ls -lh {GGUF_QUANT}
!rm -f {GGUF_F16}   # reclaim ~70 GB
!df -h /content

## 8. TurboQuant pass-through sidecar

Identity Householders + ramp codebooks. Replace with calibrated centroids (directive #4) later; for the evolve smoke test, pass-through is fine.

In [ ]:
!python {QC_DIR / 'scripts' / 'turboquant_encode.py'} \
    --model {GGUF_QUANT} \
    --out {SIDECAR_BIN} \
    --bits-k 8 --bits-v 8
!ls -lh {SIDECAR_BIN}

## 9. Gateway config

`gateway.port` + `gateway.auth.token` set the WebSocket endpoint. `evolve.enabled=true` + `evolve.bridge_script` tells `EvolveRuntime` to spawn the ASI-Evolve Python bridge as a sidecar at gateway startup.

In [ ]:
import json
EVOLVE_CONFIG = QC_DIR / 'config.colab.json'
cfg = {
    'gateway': {
        'port': GATEWAY_PORT,
        'bind': '127.0.0.1',
        'auth': {'mode': 'token', 'token': AUTH_TOKEN},
    },
    'provider': {
        'local': {
            'llama_server_url': f'http://127.0.0.1:{SERVER_PORT}',
            'model_path': str(GGUF_QUANT),
        }
    },
    'evolve': {
        'enabled': True,
        'sidecar_enabled': True,
        'bridge_script': str(BRIDGE_SCRIPT),
        'python': 'python3',
        'workdir': str(EVOLVE_DIR),
        'env': {'ASI_EVOLVE_PROVIDER_URL': f'http://127.0.0.1:{GATEWAY_PORT}'},
    },
}
EVOLVE_CONFIG.write_text(json.dumps(cfg, indent=2))
print(EVOLVE_CONFIG.read_text())

## 10. Start llama-server (background)

Full-layer offload, TurboQuant sidecar in pass-through mode.

In [ ]:
import subprocess, socket, time, requests
SERVER_LOG = ROOT / 'llama-server.log'
server_cmd = [
    str(BUILD_DIR / 'bin' / 'llama-server'),
    '-m', str(GGUF_QUANT),
    '--host', '127.0.0.1', '--port', str(SERVER_PORT),
    '--parallel', '1',
    '--ctx-size', str(CTX_SIZE),
    '-ngl', '999',
    '--turboquant', str(SIDECAR_BIN),
    '--turboquant-bits-k', '8',
    '--turboquant-bits-v', '8',
]
server_proc = subprocess.Popen(server_cmd, stdout=SERVER_LOG.open('wb'), stderr=subprocess.STDOUT)
print(f'llama-server pid={server_proc.pid}  log={SERVER_LOG}')

for _ in range(240):
    try:
        if requests.get(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=1).status_code == 200:
            print('llama-server READY'); break
    except Exception: pass
    time.sleep(1)
else:
    raise RuntimeError(f'llama-server failed  see {SERVER_LOG}')

## 11. Start the QuantClaw gateway (background)

`quantclaw gateway run --config <json>` starts the WebSocket RPC server. At boot, `EvolveRuntime::StartSidecar` spawns the ASI-Evolve Python bridge.

In [ ]:
GATEWAY_LOG = ROOT / 'gateway.log'
gateway_cmd = [
    str(BUILD_DIR / 'bin' / 'quantclaw'),
    'gateway', 'run',
    '--config', str(EVOLVE_CONFIG),
]
gateway_proc = subprocess.Popen(gateway_cmd, stdout=GATEWAY_LOG.open('wb'), stderr=subprocess.STDOUT)
print(f'gateway pid={gateway_proc.pid}  log={GATEWAY_LOG}')

# WebSocket port health: plain TCP connect probe
for _ in range(60):
    try:
        with socket.create_connection(('127.0.0.1', GATEWAY_PORT), timeout=1):
            print('gateway port OPEN'); break
    except Exception: pass
    time.sleep(1)
else:
    raise RuntimeError(f'gateway failed  see {GATEWAY_LOG}')

## 12. WebSocket RPC client

The gateway speaks a framed protocol. Each request: first `connect.hello` (auth + scopes), then `{type:'req', id, method, params}`. Tools fire through `chain.execute`.

In [ ]:
import json, itertools, websocket

class GatewayRpc:
    def __init__(self, url, token, timeout=300):
        self.ws = websocket.create_connection(url, timeout=timeout)
        self._ids = itertools.count(1)
        hello = self._send('connect.hello', {
            'clientName': 'colab-notebook',
            'clientVersion': '1.0.0',
            'authToken': token,
            'minProtocol': 1,
            'maxProtocol': 3,
            'role': 'operator',
            'scopes': ['operator.read', 'operator.write', 'operator.admin'],
        })
        self.hello = hello

    def _send(self, method, params):
        rid = str(next(self._ids))
        self.ws.send(json.dumps({'type': 'req', 'id': rid, 'method': method, 'params': params}))
        while True:
            msg = json.loads(self.ws.recv())
            if msg.get('type') == 'event': continue
            if msg.get('type') == 'res' and msg.get('id') == rid:
                if 'error' in msg and msg['error']:
                    raise RuntimeError(f'RPC {method} error: {msg["error"]}')
                return msg.get('payload') or msg.get('result') or {}

    def call(self, method, params=None):
        return self._send(method, params or {})

    def call_tool(self, tool, arguments):
        return self.call('chain.execute', {
            'name': f'notebook-{tool}',
            'error_policy': 'stop_on_error',
            'steps': [{'tool': tool, 'arguments': arguments}],
        })

    def close(self):
        try: self.ws.close()
        except: pass

rpc = GatewayRpc(GATEWAY_WS, AUTH_TOKEN)
print('hello payload:', rpc.hello)
print('tools.catalog:', json.dumps(rpc.call('tools.catalog'), indent=2)[:1500])

## 13. Kick off the evolve run

Customize `objective` / `budget` for your experiment. If `evolve_start` isn't in the tool catalog, the Python bridge didn't load  tail the gateway log.

In [ ]:
run_id = 'qwen36-turboquant-smoke-01'
result = rpc.call_tool('evolve_start', {
    'run_id': run_id,
    'objective': (
        'Write a single-file FastAPI service exposing /health and /chat that proxies to '
        'an OpenAI-compatible backend. Score on: (a) lints clean, (b) passes smoke test, (c) <80 LOC.'
    ),
    'budget': {'max_candidates': 6, 'max_wall_seconds': 900},
    'generator': {'model': HF_MODEL_ID, 'temperature': 0.6, 'max_tokens': 4096},
})
print(json.dumps(result, indent=2)[:2500])

## 14. Monitor

In [ ]:
import time
for _ in range(60):
    try:
        status = rpc.call_tool('evolve_status', {'run_id': run_id})
        print(status.get('final_result', status))
        state = (status.get('final_result') or {}).get('state')
        if state in ('completed', 'failed', 'stopped'): break
    except Exception as e:
        print(f'status error: {e}')
    time.sleep(15)

## 15. Results + logs

In [ ]:
try:
    results = rpc.call_tool('evolve_results', {'run_id': run_id, 'limit': 10})
    print(json.dumps(results, indent=2)[:4000])
except Exception as e:
    print(f'results error: {e}')

print('---- gateway log (tail) ----')
!tail -40 {GATEWAY_LOG}
print('---- server log (tail) ----')
!tail -20 {SERVER_LOG}

## 16. Teardown

In [ ]:
try: rpc.close()
except: pass
for p in [gateway_proc, server_proc]:
    p.terminate()
    try: p.wait(timeout=10)
    except: p.kill()
print('stopped')